# SmartBarn Vision — Instance Segmentation
Separate cow masks and tracking IDs; CPU-compatible. Run cells from project root.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / "smartbarn").exists() and (ROOT.parent / "smartbarn").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


Project root: D:\project-cow\smartbarn_vision-seg


In [3]:
%pip install -U "ultralytics>=8.3,<9" opencv-python numpy pandas matplotlib imageio-ffmpeg


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: D:\project-cow\cow\Scripts\python.exe -m pip install --upgrade pip


In [28]:
from smartbarn.segmentation import CowInstanceSegmenter, SegmentationConfig
cfg = SegmentationConfig(video="../videos/cow2-tri2.mp4",max_seconds=10,device="cpu",enhancement="auto",tracker="botsort.yaml",imgsz=960)
segmenter = CowInstanceSegmenter(cfg)


In [30]:
results = segmenter.process_video()
print(results)


Processed 25 frames; 13 tracked cows in latest frame
Processed 50 frames; 12 tracked cows in latest frame
Processed 75 frames; 11 tracked cows in latest frame
Processed 100 frames; 11 tracked cows in latest frame
Processed 125 frames; 11 tracked cows in latest frame
Processed 150 frames; 9 tracked cows in latest frame
Processed 175 frames; 12 tracked cows in latest frame
{'video': WindowsPath('outputs/segmentation/cow_instance_segmentation_H264.mp4'), 'raw_video': WindowsPath('outputs/segmentation/cow_instance_segmentation_mp4v.mp4'), 'csv': WindowsPath('outputs/segmentation/instances.csv'), 'frames': 199, 'detections': 2295, 'unique_track_ids': 22}


In [31]:
from IPython.display import Video, display
display(Video(filename=str(results["video"]),embed=True,width=720))


In [32]:
import pandas as pd
df=pd.read_csv(results["csv"])
display(df.head())
print("Tracker IDs:",df.track_id.nunique() if len(df) else 0)


,frame,time_seconds,track_id,confidence,pixels,x1,y1,x2,y2,enhancement
0,0,0.0,1,0.832691,7593,204.167847,190.518158,342.322693,314.382996,original
1,0,0.0,2,0.722603,11838,439.022095,346.026001,536.320557,540.355286,original
2,0,0.0,3,0.697944,15936,324.721008,340.761108,448.738586,528.761963,original
3,0,0.0,4,0.697214,13638,202.988647,728.613098,322.292236,896.814880,original
4,0,0.0,5,0.601787,7263,654.704956,387.841736,719.826172,544.671814,original


Tracker IDs: 22


In [33]:
from PIL import Image
from pathlib import Path
maps=sorted(Path(cfg.output_dir,"instance_maps").glob("*.png"))
if maps:
    labels = __import__("numpy").array(Image.open(maps[0]))
    print("Instance IDs in first frame:",__import__("numpy").unique(labels))


Instance IDs in first frame: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]


In [34]:
from matplotlib import pyplot as plt
from pathlib import Path
images=list(Path("data").rglob("*.jpg"))
if images:
    annotated, label_map, records = segmenter.predict_image(str(images[0]), save_to="outputs/segmentation/example.jpg")
    plt.figure(figsize=(8,6));plt.imshow(__import__("cv2").cvtColor(annotated,__import__("cv2").COLOR_BGR2RGB));plt.axis("off");plt.show()
else:
    print("Place a cow image under data/ to test single-image segmentation.")


Place a cow image under data/ to test single-image segmentation.
